# CUBO — Technologies applied to the room-booking agent

## 1. Introduction

CUBO is a conversational assistant for creating, listing, checking, and cancelling meeting-room bookings. The request moves from Streamlit to FastAPI, then through LangGraph tools into the service, domain, repository, and database layers. The chatbot is only an interface: the booking system remains valid without the model.

The complete project overview and component diagram are in [`doc/00-overview.md`](doc/00-overview.md).

This notebook uses code from the repository itself. It does not call OpenAI, consume tokens, or print secrets. The placeholder environment values below are used only when real values are absent, because `app.config` requires them when modules are imported.

In [ ]:
import inspect
import os
from pathlib import Path
from pprint import pprint

os.environ.setdefault("JWT_SECRET", "not-used-by-this-notebook")
os.environ.setdefault("OPENAI_API_KEY", "not-used-by-this-notebook")

assert Path("app").is_dir(), "Run the notebook from the repository root."

## 2. Pydantic — validation and tool schemas

Pydantic defines the arguments that the model may send to each tool. `CreateBookingInput` is a real schema from `app/agent/tools.py`. Its field descriptions are written for the model: they explain the accepted room names and exact date-time format.

In .NET terms, a Pydantic model is close to a DTO plus validation metadata such as FluentValidation rules. LangChain converts this model into the function schema sent to OpenAI.

In [ ]:
from app.agent.tools import CreateBookingInput

print(inspect.getsource(CreateBookingInput))
pprint(CreateBookingInput.model_json_schema())

## 3. SQLAlchemy and SQLite — persistence and concurrency

SQLAlchemy plays a role similar to EF Core. `BookingSlotModel` stores every occupied 30-minute slot and has a unique constraint on `(room_id, slot_start)`. Therefore, two bookings cannot own the same room and slot.

`BookingRepository.create()` inserts the booking and all its slots in one transaction. It does not run an availability `SELECT` first. A previous check could become stale before the insert; instead, the database decides the conflict atomically. If the constraint fails, the repository rolls back and raises `RoomNotAvailable`.

In [ ]:
from app.infrastructure.models import BookingSlotModel
from app.infrastructure.repositories.booking_repository import BookingRepository

print(inspect.getsource(BookingSlotModel))
print(inspect.getsource(BookingRepository.create))

## 4. Pure domain — `TimeRange`

The domain is plain Python and has no imports from FastAPI, SQLAlchemy, or LangGraph. `TimeRange` is a frozen dataclass, similar to an immutable C# record. It uses half-open intervals: `[start, end)`. This means one booking may start exactly when another ends.

Contiguity is not checked by a separate rule. A booking is represented by one start and one end, so a gap cannot be expressed. Contiguity is satisfied by construction.

In [ ]:
from datetime import datetime

from app.config import OFFICE_TZ
from app.domain.time_range import TimeRange

print(inspect.getsource(TimeRange.overlaps))

first = TimeRange(
    datetime(2026, 9, 7, 10, 0, tzinfo=OFFICE_TZ),
    datetime(2026, 9, 7, 11, 30, tzinfo=OFFICE_TZ),
)
touching = TimeRange(
    datetime(2026, 9, 7, 11, 30, tzinfo=OFFICE_TZ),
    datetime(2026, 9, 7, 12, 0, tzinfo=OFFICE_TZ),
)
overlapping = TimeRange(
    datetime(2026, 9, 7, 11, 0, tzinfo=OFFICE_TZ),
    datetime(2026, 9, 7, 12, 0, tzinfo=OFFICE_TZ),
)

result = {
    "10:00-11:30 vs 11:30-12:00": first.overlaps(touching),
    "10:00-11:30 vs 11:00-12:00": first.overlaps(overlapping),
}
assert result == {
    "10:00-11:30 vs 11:30-12:00": False,
    "10:00-11:30 vs 11:00-12:00": True,
}
result

## 5. FastAPI — authentication and dependency injection

FastAPI exposes the HTTP boundary and injects dependencies with `Depends`, similar to resolving services from the ASP.NET Core dependency-injection container. `get_current_user` reads the bearer token, decodes its `user_id`, and loads that user from the database.

The booking endpoint receives the authenticated user object and uses `current_user.id`. A client cannot provide `user_id` in the request body, query string, or a custom header.

In [ ]:
from app.api.deps import get_current_user
from app.api.routes.bookings import list_my_bookings

print(inspect.getsource(get_current_user))
print(inspect.getsource(list_my_bookings))

## 6. LangGraph — the agent loop

LangGraph models the agent as a small state machine with two nodes. The `agent` node asks the model what to do. If the response contains tool-call intent, `tools_condition` sends it to the `tools` node. After the tool runs, the result returns to `agent` so the model can produce the final answer or request another tool.

The model never executes Python or SQL. It only returns a structured intention; `ToolNode` executes the registered function. The next cell builds the graph and prints its Mermaid representation, but it does **not** invoke the graph or contact OpenAI.

In [ ]:
from app.agent.graph import build_graph
from app.infrastructure.database import init_db
from app.infrastructure.repositories.booking_repository import BookingRepository
from app.services.booking_service import BookingService
from sqlalchemy import create_engine
from sqlalchemy.orm import Session

print(inspect.getsource(build_graph))

notebook_engine = create_engine("sqlite:///:memory:")
init_db(notebook_engine)
notebook_session = Session(bind=notebook_engine)
notebook_service = BookingService(BookingRepository(notebook_session))
graph = build_graph(notebook_service, user_id=1)
print(graph.get_graph().draw_mermaid())

## 7. Tool calling — `user_id` injected as a closure

`build_tools(service, user_id)` receives the authenticated identity from the API. Every tool closes over that value, so the model never sees or chooses it. For example, `list_my_bookings()` has no arguments and internally calls `service.list_my_bookings(user_id)`.

This is important because prompt instructions are not a security boundary. Removing `user_id` from the tool schema makes acting as another user impossible even when the model produces an incorrect tool call.

In [ ]:
from app.agent.tools import build_tools

print(inspect.getsource(build_tools))

tools = build_tools(notebook_service, user_id=42)
tool_schemas = {built_tool.name: built_tool.args for built_tool in tools}
assert all("user_id" not in schema for schema in tool_schemas.values())
pprint(tool_schemas)

notebook_session.close()
notebook_engine.dispose()

## 8. Guardrails — prompt, domain, database, and input limits

The project uses several guardrail levels with different responsibilities:

1. The system prompt guides the conversation, but it is not trusted for enforcement.
2. The service calls pure domain rules before touching the database.
3. SQLite guarantees slot uniqueness even under concurrent requests.
4. Pydantic rejects oversized messages and histories before OpenAI sees them.
5. LangGraph has a recursion limit so a tool loop cannot consume tokens forever.

This is defense in depth: each layer protects the rule that it can enforce reliably.

In [ ]:
from app.agent.graph import _system_prompt
from app.api.routes.chat import ChatRequest, chat
from app.config import (
    AGENT_RECURSION_LIMIT,
    MAX_HISTORY_MESSAGES,
    MAX_MESSAGE_LENGTH,
)
from app.services.booking_service import BookingService

prompt = _system_prompt()
assert "server-side services are responsible for enforcing" in prompt

print("Domain validation:")
print(inspect.getsource(BookingService._validate_booking_request))
print("Database constraint:", BookingSlotModel.__table_args__)
print("Input limits:", MAX_MESSAGE_LENGTH, MAX_HISTORY_MESSAGES)
print("Graph recursion limit:", AGENT_RECURSION_LIMIT)
pprint(ChatRequest.model_json_schema())
print(inspect.getsource(chat))

## 9. Evaluation — golden conversational cases

Unit tests verify deterministic domain and application behavior. Agent evaluations answer a different question: does the model choose the correct tools and arguments across a conversation? Each golden case stores setup data, user turns, and assertions over the actual tool calls captured from the LangGraph state. Dates are calculated relative to the execution date so the cases do not expire.

The real-model suite is kept outside `tests/` because it consumes OpenAI tokens and is not deterministic. The current 16-case suite was run three times with each model:

| Model | Results | Total |
|---|---|---:|
| `gpt-5.6-terra` | 16/16, 16/16, 16/16 | 48/48 (100%) |
| `gpt-4o-mini` | 15/16, 15/16, 15/16 | 45/48 (93.8%) |

The smaller model failed the exact three-hour boundary in all three runs. This was the concrete reason for selecting Terra for the demo. Full methodology and the cost trade-off are documented in [`evals/README.md`](evals/README.md).

In [ ]:
from evals.cases import CASES

golden_case = next(
    case
    for case in CASES
    if case["name"] == "three_hour_booking_is_accepted"
)
pprint(golden_case)

# Running the full suite is intentionally not done in this notebook.
# It requires a real OPENAI_API_KEY and consumes tokens:
# python -m evals.run